# Imports and Setup


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "pyyaml", "-q"], check=True)
print("Ultralytics and YAML support are ready.")


Ultralytics and YAML support are ready.


In [ ]:
from pathlib import Path

sam2_repo_dir = "/home/vteam5/sam2"
if not Path(sam2_repo_dir).exists():
    !git clone https://github.com/facebookresearch/sam2.git "{sam2_repo_dir}" -q
else:
    print("SAM 2 repository already exists.")

%cd /home/vteam5/sam2
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python-headless", "matplotlib", "seaborn", "-q"], check=True)
print("SAM 2 and plotting tools are ready.")


SAM 2 repository already exists.
/home/vteam5/sam2
SAM 2 and plotting tools are ready.


In [ ]:
import os
import sys
import json
import random
import shutil
import xml.etree.ElementTree as ElementTree
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

from ultralytics import YOLO

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


### GPU Check


In [ ]:
gpu_available = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if gpu_available else "CPU"
compute_device = "cuda" if gpu_available else "cpu"

print(f"GPU available: {gpu_available}")
print(f"Active device: {device_name}")


GPU available: True
Active device: Quadro RTX 4000


## Extract Dataset and Create Paths


In [ ]:
project_root = Path("/home/vteam5/multispectral_pedestrian_ensemble")
llvip_dir = project_root / "llvip_dataset"
model_checkpoints_dir = project_root / "model_checkpoints"
processed_dataset_dir = project_root / "processed_dataset"
results_dir = project_root / "results"
detection_results_dir = project_root / "results_detection"
ensemble_results_dir = project_root / "results_ensemble_v2"

for directory in [project_root, llvip_dir, model_checkpoints_dir,
                  processed_dataset_dir, results_dir,
                  detection_results_dir, ensemble_results_dir]:
    directory.mkdir(parents=True, exist_ok=True)

llvip_infrared_train_dir = llvip_dir / "infrared" / "train"
llvip_visible_train_dir = llvip_dir / "visible" / "train"
llvip_annotations_dir = llvip_dir / "Annotations"

llvip_zip = project_root / "LLVIP.zip"
if not llvip_zip.exists():
    raise FileNotFoundError(f"Upload LLVIP.zip to {project_root} before running this notebook.")

if not llvip_infrared_train_dir.exists():
    print("Extracting LLVIP archive. This only runs when the dataset is missing.")
    !unzip -q "{llvip_zip}" -d "{llvip_dir}"

    for expected in ["infrared", "visible", "Annotations"]:
        if not (llvip_dir / expected).exists():
            nested = list(llvip_dir.glob(f"*/{expected}"))
            if nested:
                nested_root = nested[0].parent
                for item in nested_root.iterdir():
                    shutil.move(str(item), str(llvip_dir / item.name))
                nested_root.rmdir()
                break
    print("Dataset extraction finished.")
else:
    print("LLVIP dataset already extracted.")


LLVIP dataset already extracted.


## Path Configuration


In [ ]:
data_yaml_path = processed_dataset_dir / "data.yaml"

if data_yaml_path.exists():
    with open(data_yaml_path) as f:
        data_config = yaml.safe_load(f)

    print("data.yaml found:")
    for key, value in data_config.items():
        print(f"  {key}: {value}")
else:
    print(f"data.yaml will be created during preprocessing: {data_yaml_path}")

train_image_dir = processed_dataset_dir / "train" / "images"
val_image_dir = processed_dataset_dir / "val" / "images"
n_train_saved = len(list(train_image_dir.glob("*.jpg"))) if train_image_dir.exists() else 0
n_val_saved = len(list(val_image_dir.glob("*.jpg"))) if val_image_dir.exists() else 0
print(f"Saved training images: {n_train_saved}")
print(f"Saved validation images: {n_val_saved}")


data.yaml found:
  path: /home/vteam5/multispectral_pedestrian_ensemble/processed_dataset
  train: train/images
  val: val/images
  nc: 1
  names: ['Pedestrian']
Saved training images: 10221
Saved validation images: 1804
